# LPSE-X Inference & Explainability

Find IT! 2026 â€” Track C Phase 2

This notebook showcases the offline inference and XAI pipeline:
1. Load ONNX-format model for fast CPU inference
2. Load native `.ubj` model for SHAP explanations
3. Explain individual procurement records
4. Generate Bahasa Indonesia narratives
5. Produce counterfactual recommendations

**Runs fully offline, CPU-only.**

In [3]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from src.model import (
    load_model, load_onnx_model, load_test_artifacts,
    MODELS_DIR, CLASS_NAMES, apply_temperature
)
from src.explain import (
    get_explainer, explain_single,
    generate_counterfactuals
)
from src.narrative import (
    generate_narrative,
    generate_counterfactual_narrative,
    generate_full_report
)

print('All imports OK â€” running offline, CPU-only')

ModuleNotFoundError: No module named 'sklearn'

## 1. Load Models

- **ONNX model** â†’ fast inference path
- **Native .ubj** â†’ SHAP explanations (TreeExplainer requires native format)

In [ ]:
# Load native model for SHAP
model = load_model()
explainer = get_explainer(model)
print(f'Native model: {model.num_boosted_rounds()} trees')

# Load calibration
calibration = json.loads((MODELS_DIR / 'calibration.json').read_text())
print(f'Calibration: enabled={calibration["enabled"]}')

# Load imputation values
imputation = json.loads((MODELS_DIR / 'imputation_values.json').read_text())
print(f'Imputation values: {len(imputation)} features')

## 2. Select a Sample Record

In [ ]:
test_features, test_labels = load_test_artifacts()
print(f'Test set: {len(test_features)} records')

# Pick a sample
sample_idx = 42
sample_row = test_features.iloc[[sample_idx]]
sample_label = test_labels.iloc[sample_idx]
print(f'\nSample {sample_idx}: heuristic label = {sample_label} ({CLASS_NAMES[sample_label]})')

## 3. Inference + Explanation

Uses `explain_single()` which returns the canonical `factors` contract.

In [ ]:
explanation = explain_single(
    sample_row,
    model=model,
    explainer=explainer,
    calibration=calibration,
    top_k=5
)

print(f'Predicted: {explanation["predicted_label"]}')
print(f'Probability: {explanation["probability"]:.4f}')
print(f'All probabilities: {explanation["probabilities"]}')
print(f'\nTop {len(explanation["factors"])} factors:')
for f in explanation['factors']:
    print(f'  {f["feature"]:30s} SHAP={f["shap_value"]:+.4f}  ({f["direction"]})')

## 4. Bahasa Indonesia Narrative

In [ ]:
from IPython.display import Markdown, display

narrative = generate_narrative(explanation)
display(Markdown(narrative))

## 5. Counterfactual Recommendations

Attempts DiCE (timeboxed), falls back to SHAP-based suggestions.

In [ ]:
cf_result = generate_counterfactuals(
    sample_row, explanation, model=model, target_class=0
)

print(f'Method: {cf_result["method"]}')
print(f'Success: {cf_result["success"]}')

cf_narrative = generate_counterfactual_narrative(cf_result['suggestions'])
display(Markdown(cf_narrative))

## 6. Full Report

In [ ]:
full_report = generate_full_report(explanation, cf_result['suggestions'])
display(Markdown(full_report))

## 7. Batch Inference Demo

Fast batch inference on multiple records.

In [ ]:
# Batch predict on first 10 test records
batch = test_features.head(10)
dmatrix = xgb.DMatrix(batch)
probs = model.predict(dmatrix)

if calibration.get('enabled'):
    probs = apply_temperature(probs, calibration['temperature'])

preds = np.argmax(probs, axis=1)

results = pd.DataFrame({
    'predicted_class': preds,
    'predicted_label': [CLASS_NAMES[p] for p in preds],
    'confidence': probs.max(axis=1),
    'actual_label': test_labels.head(10).values,
})
display(results)

---

**Disclaimer:** Analisis ini didasarkan pada indikator risiko heuristik, bukan hasil investigasi forensik. Label risiko bersifat indikatif dan tidak menunjukkan adanya kecurangan yang terkonfirmasi.